In [1]:
# Tempo Sample Example Notebook
# Welcome! This notebook is intended to demonstrate the use of Tempo's Python API.
# It contains examples of:
#  - Streaming camera data
#  - Manipulating the scene at runtime
#  - Adding simulated cameras

In [14]:
# Tempo imports
import tempo_sim as tempo
import tempo_sim.tempo_core as tc
import tempo_sim.tempo_world as tw
import tempo_sim.tempo_sensors as ts
import tempo_sim.TempoImageUtils as tiu
import tempo_sim.TempoCore.Geometry_pb2 as Geometry

# Other imports
import asyncio
import cv2
import io
import math
import numpy as np
import random
import time

In [15]:
# Conecta ao Unreal que está rodando localmente
await tempo.set_server(address="127.0.0.1", port=10001)

atores = await tw.get_all_actors()

meus_robos = [
    ator for ator in atores.actors
    if "MyStreetSweeper" in ator.name or "MyStreetSweeper" in ator.actor_type
]

for ator in meus_robos:
    print(f"Nome: {ator.name} | Classe: {ator.actor_type}")

Nome: BP_MyStreetSweeper | Classe: BP_MyStreetSweeper_C


In [16]:
# pegando a posicao do robo
robo = meus_robos[0].name
posicao = await tw.get_actor_location(actor=robo)

print(
    f"Robô encontrado: {robo}\n"
    f"Posição: x={posicao.location.x:.2f}, "
    f"y={posicao.location.y:.2f}, "
    f"z={posicao.location.z:.2f}"
)

Robô encontrado: BP_MyStreetSweeper
Posição: x=-3.50, y=8.90, z=0.00


In [17]:
# pedindo pro robo se mover

import asyncio
import math
from tempo_sim.TempoCore.Geometry_pb2 import Vector

robo = meus_robos[0].name

# Lê posição e orientação atuais
estado = await tw.get_actor_transform(actor=robo)
inicio = estado.transform.location
yaw = estado.transform.rotation.y  # radianos

# “Frente” do robô, usando sua orientação atual
destino_x = inicio.x + 3.0 * math.cos(yaw)
destino_y = inicio.y + 3.0 * math.sin(yaw)
destino_z = inicio.z

passos = 30

for passo in range(1, passos + 1):
    progresso = passo / passos

    await tw.set_actor_location(
        actor=robo,
        location=Vector(
            x=inicio.x + (destino_x - inicio.x) * progresso,
            y=inicio.y + (destino_y - inicio.y) * progresso,
            z=destino_z,
        ),
    )

    confirmado = await tw.get_actor_location(actor=robo)
    print(
        f"{passo:02}/{passos} | "
        f"x={confirmado.location.x:.2f}, "
        f"y={confirmado.location.y:.2f}, "
        f"z={confirmado.location.z:.2f}"
    )

    await asyncio.sleep(0.1)

print("Movimento concluído: 3 metros para frente.")

01/30 | x=-3.40, y=8.90, z=0.00
02/30 | x=-3.30, y=8.90, z=0.00
03/30 | x=-3.20, y=8.90, z=0.00
04/30 | x=-3.10, y=8.90, z=0.00
05/30 | x=-3.00, y=8.90, z=0.00
06/30 | x=-2.90, y=8.90, z=0.00
07/30 | x=-2.80, y=8.90, z=0.00
08/30 | x=-2.70, y=8.90, z=0.00
09/30 | x=-2.60, y=8.90, z=0.00
10/30 | x=-2.50, y=8.90, z=0.00
11/30 | x=-2.40, y=8.90, z=0.00
12/30 | x=-2.30, y=8.90, z=0.00
13/30 | x=-2.20, y=8.90, z=0.00
14/30 | x=-2.10, y=8.90, z=0.00
15/30 | x=-2.00, y=8.90, z=0.00
16/30 | x=-1.90, y=8.90, z=0.00
17/30 | x=-1.80, y=8.90, z=0.00
18/30 | x=-1.70, y=8.90, z=0.00
19/30 | x=-1.60, y=8.90, z=0.00
20/30 | x=-1.50, y=8.90, z=0.00
21/30 | x=-1.40, y=8.90, z=0.00
22/30 | x=-1.30, y=8.90, z=0.00
23/30 | x=-1.20, y=8.90, z=0.00
24/30 | x=-1.10, y=8.90, z=0.00
25/30 | x=-1.00, y=8.90, z=0.00
26/30 | x=-0.90, y=8.90, z=0.00
27/30 | x=-0.80, y=8.90, z=0.00
28/30 | x=-0.70, y=8.90, z=0.00
29/30 | x=-0.60, y=8.90, z=0.00
30/30 | x=-0.50, y=8.90, z=0.00
Movimento concluído: 3 metros para frent